# 📝 RAG 파이프라인 과제 LV1(기초) — 파싱·청킹·색인·검색·지표

> 이 단원에서 배운 **PDF 파싱·출처 확인·청킹·임베딩 색인·Top-K 검색·메타데이터 필터·검색 품질 지표** 를 **한 문제에 하나씩** 확인하는 과제입니다.

## 풀이 방법
1. 위에서부터 **준비 셀**(제공 코드)을 먼저 실행하세요.
2. 각 문제의 **답안 셀**(`# 여기에 코드를 작성하세요`)에 코드를 채웁니다.
3. 바로 아래 **자가채점 셀**(`# [자가채점]`)을 실행해 `✅ 통과!` 가 뜨면 성공이에요.
4. 막히면 `힌트` 를 펼쳐 보세요.

## 다루는 자료
개인정보보호위원회가 배포한 **개인정보 질의응답 모음집**(2025년 12월)입니다. 실제 민원과 답변을 모아 놓은 130쪽짜리 안내서예요.

- `data/qna_docs.csv` — 질의응답 **93건**. 한 건이 원본 PDF 의 **한 쪽**입니다.
  - `id` 문서 번호 · `분야` 8종 · `조항` 개인정보 보호법 조문 · `쪽` 원본 PDF 쪽 번호 · `질문` · `본문`
- `data/원본/개인정보_질의응답_모음집.pdf` — 원본 PDF. 1·2번에서 직접 파싱합니다.

`쪽` 이 있으니 **답변에 근거를 붙이고, 그 근거를 실제로 열어 확인**할 수 있습니다. 이 구조를 이 과제 내내 씁니다.

이 노트북은 API 키가 없어도 끝까지 돌아갑니다(생성은 LV2 에서 합니다).

화이팅!

---
## 데이터 살펴보기
아래 셀은 **실행만** 하면 됩니다.

In [ ]:
# [제공 코드] 질의응답 모음집을 불러와 훑어봅니다
import pandas as pd

qna_docs = pd.read_csv('data/qna_docs.csv')
print(f'문서 수: {len(qna_docs)}')
print(f"쪽 범위: {qna_docs['쪽'].min()} ~ {qna_docs['쪽'].max()}")
print('분야별 건수:')
print(qna_docs['분야'].value_counts())
display(qna_docs[['id', '분야', '조항', '쪽', '질문']].head())

## 1. 원본 PDF 에서 쪽 번호와 함께 글자 뽑기
**배경**: RAG 의 첫 단계는 문서에서 글자를 뽑는 일입니다. 이때 **어느 쪽에서 나왔는지**를 함께 챙겨야 나중에 답변에 근거를 붙일 수 있습니다. `pymupdf4llm` 은 `page_chunks=True` 를 주면 쪽마다 딕셔너리 하나씩을 돌려주고, 그 안에 본문과 쪽 번호가 함께 들어 있습니다.

**요구사항**:
- `pymupdf4llm.to_markdown(경로, page_chunks=True, pages=[29, 30, 31])` 로 `data/원본/개인정보_질의응답_모음집.pdf` 의 **세 쪽만** 파싱하세요.
- 각 원소의 `['metadata']['page_number']` 를 순서대로 모아 리스트 **`page_nums`** 에 담으세요.
- 각 원소의 `['text']` 를 순서대로 모아 리스트 **`page_texts`** 에 담으세요.

**예시**: `page_nums` 는 `[30, 31, 32]`, `page_texts` 는 길이 3 인 리스트입니다. `page_texts[0]` 에 `'안내판'` 이라는 말이 들어 있습니다.

> `pages` 에 넘긴 값과 `page_nums` 가 왜 다른지 **직접 출력해 확인**하세요. 두 값이 세는 기준이 다릅니다. 이걸 모르면 인용한 쪽이 한 쪽씩 밀립니다.

<details><summary>힌트</summary>

```text
접근방법:
- 쪽마다 딕셔너리 하나가 돌아온다. 반복문으로 필요한 값만 꺼내 두 리스트에 모은다.

세부구현:
1. pymupdf4llm 을 임포트하고 page_chunks 와 pages 를 준 채로 파싱한다.
2. 결과를 반복하며 metadata 안의 쪽 번호를 page_nums 에 모은다.
3. 같은 반복에서 본문 문자열을 page_texts 에 모은다.
4. 넘긴 pages 와 돌아온 page_nums 를 함께 출력해 두 값이 어떻게 다른지 확인한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert page_nums == [30, 31, 32], 'page_number 를 순서대로 모았는지 확인하세요'
assert len(page_texts) == 3
assert '안내판' in page_texts[0], '첫 쪽 본문이 아닙니다'
# 세 쪽이 서로 다른 내용인지 — 같은 쪽을 세 번 담지 않았는지 확인한다
assert len(set(page_texts)) == 3
# 손으로 적은 글자가 아니라 파싱 결과인지 — 쪽마다 수백 자가 있어야 한다
assert all(len(t) > 200 for t in page_texts)
print('✅ 통과!')

## 2. 인용한 쪽을 실제로 열어 확인하기
**배경**: RAG 가 "30쪽에 이렇게 적혀 있습니다" 라고 답해도, 그 쪽을 열어 보지 않으면 사실인지 알 수 없습니다. 표에 적힌 쪽 번호로 **원본을 되짚어 확인하는 도구**를 만들어 둡니다.

**요구사항**:
- 함수 **`check_quote(doc_id, quote)`** 를 만드세요.
  1. `qna_docs` 에서 `id` 가 `doc_id` 인 행을 찾아 그 행의 **`쪽`** 값을 얻습니다.
  2. `data/원본/개인정보_질의응답_모음집.pdf` 에서 **그 쪽 하나만** `page_chunks=True` 로 파싱합니다(1번에서 확인한 `pages` 기준을 그대로 적용하세요).
  3. 그 쪽의 `['text']` 안에 `quote` 문자열이 있으면 **`True`**, 없으면 **`False`** 를 돌려줍니다.

**예시**:
- `check_quote('q1', 'ID 등으로 이미 보유한')` → `True` (`q1` 은 13쪽)
- `check_quote('q1', '영상정보처리기기')` → `False`

<details><summary>힌트</summary>

```text
접근방법:
- 표에서 쪽 번호를 찾고, 그 쪽만 파싱해, 문자열이 들어 있는지 in 으로 본다.

세부구현:
1. qna_docs 에서 id 열이 doc_id 와 같은 행을 골라 쪽 값을 정수로 꺼낸다.
2. 1번에서 확인한 기준에 맞춰 pages 에 넘길 값을 만든다(쪽 번호 그대로가 아니다).
3. 그 한 쪽을 파싱해 첫 원소의 본문 문자열을 얻는다.
4. quote 가 그 문자열 안에 있는지 in 으로 검사해 참·거짓을 반환한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert check_quote('q1', 'ID 등으로 이미 보유한') is True
assert check_quote('q1', '영상정보처리기기') is False
# 지문에 없던 다른 문서로 다시 호출한다 — q1 만 맞게 짠 답은 여기서 걸린다
assert check_quote('q16', '대표적인 안내판만 설치해도 됩니다') is True
assert check_quote('q16', 'ID 등으로 이미 보유한') is False, \
    '다른 쪽의 문장까지 True 가 나옵니다 — 그 문서의 쪽만 파싱했는지 확인하세요'
# 원본 PDF 를 실제로 읽었는지 — 아래 문구는 쪽 머리글이라 CSV 본문에는 없다
assert check_quote('q22', '개인정보 질의응답 모음집') is True, \
    'CSV 의 본문이 아니라 원본 PDF 의 그 쪽을 파싱해야 합니다'
print('✅ 통과!')

## 3. 오버랩 청킹 함수 만들기
**배경**: 문서를 글자 수로 뚝뚝 자르면 경계에서 문장이 끊깁니다. 앞 조각의 끝 일부를 다음 조각이 **겹쳐** 갖게 하면 경계 문맥이 살아남습니다.

**요구사항**:
- 함수 **`chunk_overlap(text, size, overlap)`** 를 만드세요.
- 각 조각의 길이는 `size` 이고, 다음 조각의 시작 위치는 **`size - overlap`** 글자씩 이동합니다.
- 마지막 조각은 `size` 보다 짧을 수 있습니다.

**예시**: `chunk_overlap('abcdefgh', 4, 2)` → `['abcd', 'cdef', 'efgh', 'gh']`

<details><summary>힌트</summary>

```text
접근방법:
- 이동 간격을 크기에서 겹침을 뺀 값으로 정하고, 그 간격으로 시작 위치를 훑는다.

세부구현:
1. 이동 간격을 size 에서 overlap 을 뺀 값으로 계산한다.
2. 0 부터 글자 수까지 그 간격으로 시작 위치를 만든다.
3. 각 시작 위치에서 size 글자를 잘라 리스트에 모아 반환한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert chunk_overlap('abcdefgh', 4, 2) == ['abcd', 'cdef', 'efgh', 'gh']
assert chunk_overlap('abcdef', 3, 1) == ['abc', 'cde', 'ef']
# 겹침이 0 이면 고정 크기와 같아져야 한다 — step 을 size 로 잘못 고정하면 여기서 걸린다
assert chunk_overlap('abcdefgh', 4, 0) == ['abcd', 'efgh']
# 겹침을 키우면 조각 수가 늘어난다
assert len(chunk_overlap('a' * 100, 20, 10)) > len(chunk_overlap('a' * 100, 20, 5))
print('✅ 통과!')

## 4. 세 전략으로 같은 질의응답을 잘라 비교하기
**배경**: 청킹 전략은 하나가 아닙니다. **글자 수로 자르기**·**겹쳐 자르기**·**문단 경계에서 자르기** 는 같은 문서를 서로 다른 개수·모양으로 나눕니다. 어느 쪽이 맞는지는 문서를 보고 정해야 하므로, 먼저 **재 보는 습관**을 들입니다.

아래 준비 셀이 `chunk_fixed` 와 `chunk_paragraph` 를 제공합니다. `chunk_overlap` 은 3번에서 만든 것을 그대로 씁니다.

**요구사항**:
- `qna_docs` 에서 `id` 가 **`'q70'`** 인 문서의 `본문` 을 꺼내 변수 **`long_text`** 에 담으세요.
- 세 전략을 모두 **`size=400`** 로 적용하세요(겹쳐 자르기는 `overlap=100`).
- 각 전략의 **조각 개수**를 딕셔너리 **`chunk_counts`** 에 담으세요. 키는 **`'fixed'`·`'overlap'`·`'paragraph'`** 입니다.

**예시**: `chunk_counts` 는 `{'fixed': 3, 'overlap': ..., 'paragraph': ...}` 처럼 세 개의 정수를 담은 딕셔너리입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 문서 하나의 본문을 꺼내 세 함수에 차례로 넘기고, 결과 리스트의 길이를 센다.

세부구현:
1. 표에서 id 가 문제에 적힌 문서 번호인 행을 골라 본문 열을 문자열로 꺼낸다.
2. 세 함수에 각각 같은 크기를 넘겨 조각 리스트를 얻는다(겹쳐 자르기만 인자가 하나 더).
3. 각 리스트의 길이를 세 키에 맞춰 딕셔너리로 만든다.
4. 조각 개수와 각 조각의 길이를 출력해 세 전략의 차이를 확인한다.
```

</details>

In [ ]:
# [제공 코드] 나머지 두 청킹 함수 — 이 강의에서 만든 것을 그대로 가져왔습니다
def chunk_fixed(text, size):
    """고정 크기(글자 수)로 자른다."""
    return [text[i:i + size] for i in range(0, len(text), size)]

def chunk_paragraph(text, size):
    """빈 줄로 나뉜 문단을 순서대로 모으되, size 를 넘기 직전에 끊는다."""
    chunks, cur = [], ''
    for para in [p.strip() for p in text.split('\n\n') if p.strip()]:
        if cur and len(cur) + len(para) > size:
            chunks.append(cur)
            cur = para
        else:
            cur = f'{cur}\n{para}' if cur else para
    if cur:
        chunks.append(cur)
    return chunks

print('chunk_fixed · chunk_paragraph 준비 완료')

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(long_text, str) and len(long_text) > 1000
assert chunk_counts == {'fixed': 3, 'overlap': 4, 'paragraph': 3}, \
    '세 전략을 모두 size=400 로 적용했는지 확인하세요'
# 손으로 적은 숫자가 아니라 함수 결과인지 — 같은 재료로 다시 세어 대조한다
assert chunk_counts['overlap'] == len(chunk_overlap(long_text, 400, 100))
assert chunk_counts['paragraph'] == len(chunk_paragraph(long_text, 400))
# 겹쳐 자르면 같은 크기라도 조각이 더 많아진다
assert chunk_counts['overlap'] > chunk_counts['fixed']
print('✅ 통과!')

---
## 검색 준비
이제 문서를 벡터로 바꿔 검색합니다. 아래 두 셀은 **실행만** 하세요(모델을 내려받느라 처음엔 잠시 걸립니다).

LV1 에서는 문서를 자르지 않고 **한 건을 통째로** 색인합니다. 청킹한 조각을 색인하는 일은 LV2 에서 합니다.

In [ ]:
# [제공 코드] 임베딩 모델 준비 — 지난 시간에 쓴 한국어 문장 임베딩 모델입니다(불러오는 데 잠시 걸립니다).
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('jhgan/ko-sroberta-multitask')
print('임베딩 모델 준비 완료 (768차원)')

In [ ]:
# [제공 코드] 색인에 넣을 재료를 준비합니다
import chromadb

qna_ids = qna_docs['id'].tolist()
qna_texts = qna_docs['본문'].tolist()

# 메타데이터에 분야·조항·쪽을 함께 싣습니다. 색인할 때 안 실으면 검색 결과에
# '어느 분야 몇 쪽에서 나왔는지'를 붙일 방법이 없습니다.
qna_metas = [{'doc_id': r['id'], 'field': r['분야'], 'article': r['조항'], 'page': int(r['쪽'])}
             for _, r in qna_docs.iterrows()]
print(f'준비된 문서 수: {len(qna_ids)}')
print(f'첫 문서의 메타데이터: {qna_metas[0]}')

## 5. 임베딩 색인 만들기
**배경**: 의미로 검색하려면 문서를 임베딩해 벡터DB 에 넣어야 합니다.

**요구사항**:
- `qna_texts` 를 `embed_model` 로 임베딩하세요 — 코사인 거리로 재려면 `normalize_embeddings` 옵션을 켜야 합니다.
- `chromadb.EphemeralClient()` 로 컬렉션을 하나 만들어 변수 **`qna_collection`** 에 담으세요. 이름은 **`'qna_lv1_docs'`**, 거리 방식은 **코사인**입니다(`metadata` 의 `'hnsw:space'` 키).
- 그 컬렉션에 `qna_ids`·임베딩·`qna_texts`·`qna_metas` 를 함께 적재하세요(본문은 `documents`, 메타데이터는 `metadatas` 자리에 들어갑니다).

**예시**: 적재가 끝나면 `qna_collection.count()` 가 **93** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 임베딩 결과는 넘파이 배열이므로 적재할 때는 파이썬 리스트로 바꿔 넣는다.

세부구현:
1. 문서 리스트를 임베딩 모델로 인코딩한다(정규화 옵션을 켜 코사인용 단위벡터로).
2. 코사인 거리로 컬렉션을 만들어 변수에 담는다.
3. id·임베딩·본문·메타데이터를 함께 적재한다.
4. 적재된 개수를 출력해 확인한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert qna_collection.count() == len(qna_docs)
# 개수만 맞추면 아무 벡터나 넣어도 통과하므로, 실제로 본문을 임베딩했는지 검색으로 확인한다
probe_vec = embed_model.encode(['가명정보를 만들려면 무엇을 지워야 하나요'],
                               normalize_embeddings=True)
probe = qna_collection.query(query_embeddings=probe_vec.tolist(), n_results=1)
hit_meta = probe['metadatas'][0][0]
assert hit_meta['doc_id'] == 'q28', \
    '가명정보 질의응답(q28)이 1위로 나오지 않습니다 — qna_texts 를 그대로 임베딩했는지 확인하세요'
assert probe['documents'][0][0] in qna_texts, 'documents 에 원문이 들어가 있어야 합니다'
# 출처 표기에 쓸 분야·조항·쪽이 색인 안에 들어갔는지 확인한다
q28_row = qna_docs.set_index('id').loc['q28']
assert hit_meta.get('field') == q28_row['분야'], \
    'field 메타데이터가 없거나 다릅니다 — qna_metas 를 그대로 넘기세요'
assert hit_meta.get('page') == int(q28_row['쪽']), \
    'page 메타데이터가 없거나 정수가 아닙니다 — 출처 표기에 필요합니다'
print('✅ 통과!')

## 6. Top-K 검색
**배경**: 만든 색인에서 질문과 **가까운 문서 K개**를 찾습니다.

**요구사항**:
- 질문 **'건물에 카메라를 여러 대 달았는데 안내판은 몇 개나 붙여야 하나요'** 을 **문서와 같은 모델로** 임베딩해 `qna_collection` 에서 **상위 3개**를 검색하세요(`query` 의 `query_embeddings`·`n_results` 인자).
- 결과의 `metadatas` 첫 리스트에서 각 항목의 `doc_id` 를 모아 리스트 **`found_ids`** 에 담으세요.

**예시**: `found_ids` 는 상위 3개 문서 id 목록입니다. 안내판 질의응답 **`q16`** 이 들어 있어야 합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 질문도 문서와 같은 모델로 임베딩해야 같은 공간에서 거리를 잰다.

세부구현:
1. 질문 문장 하나를 리스트에 담아 임베딩한다(정규화 옵션 켜기).
2. 그 임베딩으로 컬렉션을 조회해 상위 3건을 받는다.
3. 결과의 메타데이터 목록에서 문서 id 만 뽑아 리스트로 만든다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(found_ids, list) and len(found_ids) == 3
assert 'q16' in found_ids, '안내판 질의응답(q16)이 상위 3개에 없습니다'
# 손으로 적은 목록이 아니라 실제 검색 결과인지 — 같은 질문을 다시 검색해 대조한다
recheck_vec = embed_model.encode(['건물에 카메라를 여러 대 달았는데 안내판은 몇 개나 붙여야 하나요'], normalize_embeddings=True)
recheck = qna_collection.query(query_embeddings=recheck_vec.tolist(), n_results=3)
assert found_ids == [m['doc_id'] for m in recheck['metadatas'][0]], \
    '실제 검색 결과와 순서·내용이 다릅니다 — 직접 검색한 결과를 담으세요'
print('✅ 통과!')

## 7. 메타데이터로 좁혀 검색하기
**배경**: 같은 말이라도 분야에 따라 답이 다릅니다. "얼마나 오래 보관하나" 는 CCTV 영상이냐 회원 정보냐에 따라 근거 조문이 완전히 달라집니다. 이럴 때 **메타데이터로 후보를 먼저 좁혀** 검색합니다.

**요구사항**:
- 질문 **'개인정보를 얼마나 오래 보관할 수 있나요'** 을 임베딩해 **같은 임베딩으로 두 번** 검색하세요(둘 다 상위 3개).
  - 조건 없이 검색한 결과의 `doc_id` 목록 → **`plain_ids`**
  - `query` 의 **`where`** 인자로 메타데이터 `field` 가 **'영상정보'** 인 것만 남겨 검색한 결과의 `doc_id` 목록 → **`filtered_ids`**
- 두 목록을 함께 출력해 어떻게 달라지는지 보세요.

**예시**: `plain_ids` 와 `filtered_ids` 는 각각 길이 3 인 리스트이고, `filtered_ids` 는 **모두 `영상정보` 분야 문서**입니다. 보관기간을 다루는 **`q22`·`q14`** 이 둘 다 들어 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 같은 질문 임베딩을 두 번 쓰되, 두 번째 조회에만 메타데이터 조건을 얹는다.

세부구현:
1. 질문을 한 번만 임베딩해 변수에 담는다.
2. 조건 없이 상위 3건을 조회해 문서 id 를 모은다.
3. 같은 임베딩으로 where 조건을 준 채 상위 3건을 조회해 문서 id 를 모은다.
4. 두 목록을 나란히 출력한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(plain_ids) == 3 and len(filtered_ids) == 3
# 필터를 걸었으면 결과가 전부 그 분야여야 한다
field_of = qna_docs.set_index('id')['분야'].to_dict()
assert all(field_of[d] == '영상정보' for d in filtered_ids), \
    'filtered_ids 에 영상정보 가 아닌 문서가 섞여 있습니다 — where 조건을 확인하세요'
# 필터가 실제로 결과를 바꿨는지 — 두 목록이 같으면 필터가 안 걸린 것이다
assert plain_ids != filtered_ids, 'where 조건이 걸리지 않았습니다'
assert not all(field_of[d] == '영상정보' for d in plain_ids), \
    'plain_ids 에도 필터가 걸려 있습니다 — 조건 없이 검색한 결과를 담으세요'
assert set(['q22', 'q14']) <= set(filtered_ids), \
    '보관기간 질의응답 두 건이 모두 들어 있어야 합니다 — 10번이 이 결과를 그대로 씁니다'
print('✅ 통과!')

## 8. 지표 — Precision@K
**배경**: 검색이 얼마나 정확한지는 말이 아니라 숫자로 재야 개선 여부를 알 수 있습니다. Precision@K 는 **가져온 상위 K개 중 정답의 비율**입니다(가져온 것이 얼마나 정확한가).

**요구사항**:
- 함수 **`precision_at_k(predicted, relevant, k)`** 를 만드세요. (상위 k개 중 `relevant` 에 든 것의 개수) / k 를 돌려줍니다.

**예시**: `predicted=['q3','q1','q7']`, `relevant=['q1','q5']`, `k=3` → 정답 1개 / 3 = **0.333…**

<details><summary>힌트</summary>

```text
접근방법:
- 상위 k개만 잘라서 정답에 든 개수를 세고 k 로 나눈다.

세부구현:
1. 예측 목록의 앞 k개만 본다.
2. 그중 정답 목록에 들어 있는 것의 개수를 센다.
3. 그 개수를 k 로 나눠 반환한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(precision_at_k(['q3', 'q1', 'q7'], ['q1', 'q5'], 3) - 1 / 3) < 1e-9
assert abs(precision_at_k(['q1', 'q5', 'q7'], ['q1', 'q5'], 2) - 1.0) < 1e-9
assert abs(precision_at_k(['q3', 'q4', 'q7'], ['q1'], 3) - 0.0) < 1e-9
# 상위 k 로 자르지 않으면 3/2 = 1.5 라는 비율이 될 수 없는 값이 나와 걸린다
assert abs(precision_at_k(['q1', 'q5', 'q7'], ['q1', 'q5', 'q7'], 2) - 1.0) < 1e-9
print('✅ 통과!')

## 9. 지표 — MRR
**배경**: 정답을 찾았는지만 보면 1위로 올린 검색기와 5위에 겨우 끼워 넣은 검색기가 같아 보입니다. MRR 은 **첫 정답이 몇 등인지**를 봅니다(1등이면 1, 2등이면 1/2 …).

**요구사항**:
- 함수 **`mrr(predicted, relevant)`** 를 만드세요. 앞에서부터 보며 **첫 정답의 등수 역수**를 돌려줍니다. 정답이 하나도 없으면 **`0.0`** 입니다.

**예시**: `predicted=['q3','q1','q7']`, `relevant=['q1','q5']` → 첫 정답 `q1` 이 2등 → 1/2 = **0.5**

<details><summary>힌트</summary>

```text
접근방법:
- 1등부터 순서대로 보며 처음으로 정답에 든 등수를 찾는다.

세부구현:
1. 등수를 1부터 세며 예측 목록을 훑는다.
2. 처음으로 정답 목록에 있는 항목을 만나면 등수의 역수를 반환한다.
3. 끝까지 만나지 못하면 0.0 을 반환한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(mrr(['q3', 'q1', 'q7'], ['q1', 'q5']) - 0.5) < 1e-9
assert abs(mrr(['q1', 'q3', 'q7'], ['q1']) - 1.0) < 1e-9
assert abs(mrr(['q3', 'q7', 'q9'], ['q1', 'q5']) - 0.0) < 1e-9
# 뒤에 정답이 더 있어도 '첫' 정답의 등수만 본다 — 마지막 정답을 쓰면 여기서 걸린다
assert abs(mrr(['q3', 'q1', 'q5'], ['q1', 'q5']) - 0.5) < 1e-9
print('✅ 통과!')

## 10. 실제 검색 결과를 네 지표로 재기
**배경**: 지금까지는 정해진 예시로 공식을 확인했습니다. 이번엔 **7번에서 실제로 검색한 결과**를 네 지표로 한꺼번에 재 봅니다. 아래 준비 셀이 나머지 두 지표(`hit_at_k`·`recall_at_k`)를 제공합니다.

**요구사항**:
- 7번의 **`filtered_ids`**(영상정보로 좁혀 찾은 상위 3개)를 예측으로, 이 질문의 정답 문서는 **`['q22', 'q14']`**(둘 다 보관기간을 다루는 질의응답) 입니다. **K=3** 으로 재세요.
- 네 값을 딕셔너리 **`scores`** 에 담으세요. 키는 **`'hit'`·`'precision'`·`'recall'`·`'mrr'`** 입니다.
- 값을 손으로 적지 말고 **반드시 함수를 호출**해 구하세요.

**예시**: `scores` 는 `{'hit': 1, 'precision': ..., 'recall': ..., 'mrr': ...}` 처럼 네 개의 수를 담은 딕셔너리입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 같은 예측·정답 목록을 네 함수에 각각 넘긴다. MRR 만 K 를 받지 않는다.

세부구현:
1. 예측 목록과 정답 목록, K 값을 준비한다.
2. 네 함수를 각각 호출해 값을 얻는다.
3. 네 값을 정해진 키로 딕셔너리에 담아 출력한다.
4. 정밀도와 재현율의 분모가 각각 무엇인지 확인한다(하나는 K, 하나는 전체 정답 수).
```

</details>

In [ ]:
# [제공 코드] 나머지 두 지표 — 이 강의에서 만든 것을 그대로 가져왔습니다
def hit_at_k(predicted, relevant, k):
    """상위 k개 중 관련 문서가 하나라도 있으면 1, 없으면 0."""
    return 1 if any(p in relevant for p in predicted[:k]) else 0

def recall_at_k(predicted, relevant, k):
    """전체 관련 문서 중 상위 k개가 찾아낸 비율."""
    hits = sum(1 for p in predicted[:k] if p in relevant)
    return hits / len(relevant)

print('hit_at_k · recall_at_k 준비 완료')

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert set(scores) == {'hit', 'precision', 'recall', 'mrr'}
# 손으로 적은 값이 아니라 함수 결과인지 — 같은 재료로 다시 계산해 대조한다
want = ['q22', 'q14']
assert scores['hit'] == hit_at_k(filtered_ids, want, 3)
assert abs(scores['precision'] - precision_at_k(filtered_ids, want, 3)) < 1e-9
assert abs(scores['recall'] - recall_at_k(filtered_ids, want, 3)) < 1e-9
assert abs(scores['mrr'] - mrr(filtered_ids, want)) < 1e-9
# 정답이 2개인데 K=3 이라 정밀도는 1 이 될 수 없고, 둘 다 찾았으므로 재현율은 1 이다
assert scores['precision'] < 1.0 and abs(scores['recall'] - 1.0) < 1e-9, \
    'filtered_ids 가 7번의 검색 결과인지 확인하세요'
print('✅ 통과!')

---
수고했어요! LV1 에서 **PDF 파싱과 출처 확인 · 청킹 · 임베딩 색인 · Top-K 검색 · 메타데이터 필터 · 네 가지 지표**를 하나씩 익혔습니다.

LV2 에서는 **파싱·청킹·색인·검색·지표를 조합**합니다. 문서를 잘라 색인하고, 질문에서 분야를 **자동으로 뽑아** 필터로 걸고, 찾은 근거에 **출처를 붙여 답변을 생성**합니다.